In [9]:
from pymatgen.ext.matproj import MPRester
from pymatgen.electronic_structure.core import Spin
import numpy as np
from scipy.optimize import curve_fit

API_KEY = "zA1YwwFoS7C8RQ3ZpVn2lK2ClRoSNTFM"  
material_id = "mp-27869" 

# Récupération de la structure de bandes
mpr = MPRester(API_KEY)
bs = mpr.get_bandstructure_by_material_id(material_id, line_mode=True)

#max et min
vbm = bs.get_vbm()
cbm = bs.get_cbm()

#calcul gap
gap = cbm["energy"] - vbm["energy"]
print(f"Bande interdite : {gap:.3f} eV")

#typde de gap
if vbm["kpoint"].frac_coords.tolist() == cbm["kpoint"].frac_coords.tolist():
    print("Type de gap : Direct")
else:
    print("Type de gap : Indirect")
    
# Dernière bande de valence
val_idx = vbm["band_index"][Spin.up][0] 
# Première bande de conduction
cond_idx = cbm["band_index"][Spin.up][0]    

#dispertion des bandes
dispersions = {"valence": {}, "conduction": {}}

#segment entre 2 points
for branch in bs.branches:
    i_start = branch["start_index"]
    i_end = branch["end_index"]
    label = f'{bs.kpoints[i_start].label} → {bs.kpoints[i_end].label}'  

    dists = bs.distance[i_start:i_end+1]  # Distance en k entre les points
    
#energie pour ce segment
e_val = bs.bands[Spin.up][val_idx][i_start:i_end+1]
e_cond = bs.bands[Spin.up][cond_idx][i_start:i_end+1]
    
#dispertion
dispersions["valence"][label] = np.ptp(e_val)
dispersions["conduction"][label] = np.ptp(e_cond)

#affichage dispertion
for band_type in ["valence", "conduction"]:
    max_disp = max(dispersions[band_type], key=dispersions[band_type].get)
    min_disp = min(dispersions[band_type], key=dispersions[band_type].get)

    print(f"\n{band_type.capitalize()} :")
    print(f"  Dispersion maximale : {max_disp} ({dispersions[band_type][max_disp]:.3f} eV)")
    print(f"  Dispersion minimale : {min_disp} ({dispersions[band_type][min_disp]:.3f} eV)")
    
#fonction parabolique
def parabole(k, a, b, c):
    return a * k**2 + b * k + c

#calcul  masse effective
def masse_effective(kpts, energies):
    popt, _ = curve_fit(parabole, kpts, energies)  # Ajustement des données
    a = popt[0]  # Coefficient a de la parabole
    
# Conversion en unités SI
    hbar = 1.055e-34      # Constante de Planck réduite (J·s)
    eV_to_J = 1.602e-19   # Conversion eV → Joules
    Å_to_m = 1e-10        # Angström → mètre

    a_SI = a * eV_to_J / (Å_to_m**2)  # a en unités SI
    m_eff = hbar**2 / (2 * a_SI)      # Formule de la masse effective

    return m_eff / 9.109e-31  # Masse réduite (par rapport à la masse de l’électron)

# Calcul de la masse effective au sommet de la bande de valence
# Choix d’une branche pour l’analyse 
vb_branch = bs.branches[0]

# Indices de début et fin du segment
i0 = vb_branch["start_index"]
i1 = vb_branch["end_index"]

# Distance en k et énergies associées
kpts = bs.distance[i0:i1+1]
e_vals = bs.bands[Spin.up][val_idx][i0:i1+1]

# On repère le sommet (maximum) de la bande de valence
idx_max = np.argmax(e_vals)

# On sélectionne 5 points autour du sommet pour l’ajustement
window = slice(max(0, idx_max-2), idx_max+3)

# Calcul de la masse effective
m_eff_val = masse_effective(kpts[window], e_vals[window])

# Affichage du résultat
print(f"\nMasse effective au sommet de la bande de valence : {m_eff_val:.2f} mₑ")



Retrieving ElectronicStructureDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Bande interdite : 1.260 eV
Type de gap : Direct

Valence :
  Dispersion maximale : L → P (0.047 eV)
  Dispersion minimale : L → P (0.047 eV)

Conduction :
  Dispersion maximale : L → P (0.603 eV)
  Dispersion minimale : L → P (0.603 eV)

Masse effective au sommet de la bande de valence : -0.00 mₑ
